In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()  # notebook is inside notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added:", PROJECT_ROOT)


✅ Project root added: C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4


In [3]:
import hashlib
from typing import Dict, Any, List


In [4]:
from agents.cartographer import graphrag_cartographer
from agents.architect import architect_ensemble
from agents.verifier import execution_verifier
from agents.critic import semantic_verifier
from agents.orchestrator import reflex_orchestrator

print("✅ All agents imported")


🔥 cartographer.py LOADED
✅ All agents imported


In [5]:
import importlib
import agents.orchestrator
importlib.reload(agents.orchestrator)

from agents.orchestrator import reflex_orchestrator

print("✅ Orchestrator reloaded")


✅ Orchestrator reloaded


In [6]:
from agents.orchestrator import reflex_orchestrator
import inspect
print(inspect.signature(reflex_orchestrator))



(question: str, schema_text: str, db_path: pathlib.Path, graph, schema_texts, schema_ids, embedder, faiss_index, doc_tokens, df_stats, critic_llm_param, llm_large, llm_medium, max_retries: int = 5, min_confidence: float = 0.6) -> Dict[str, Any]


In [7]:
import networkx as nx
import faiss
from agents.embeddings import HFTextEmbedder


In [8]:
import networkx as nx
import faiss
from agents.embeddings import HFTextEmbedder
from agents.cartographer import build_df

# ---- Schema texts (example, later auto-built from DB) ----
schema_texts = [
    "student table with columns student_id, name",
    "course table with columns course_id, title",
    "enrollment table with columns student_id, course_id"
]

schema_ids = ["student", "course", "enrollment"]

# ---- Graph ----
graph = nx.Graph()
for t in schema_ids:
    graph.add_node(t)

graph.add_edge("student", "enrollment")
graph.add_edge("course", "enrollment")

# ---- Embedder ----
embedder = HFTextEmbedder()

embeddings = embedder.encode(schema_texts)

# ---- FAISS ----
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

# ---- Tokens + DF ----
doc_tokens = [t.lower().split() for t in schema_texts]
df_stats = build_df(doc_tokens)

print("✅ Cartographer state ready")




✅ Cartographer state ready


In [9]:
from agents.cartographer import build_df

df_stats = build_df(doc_tokens)

print("✅ DF size:", len(df_stats))


✅ DF size: 11


In [10]:
def cartographer_adapter(question: str) -> Dict[str, Any]:
    return graphrag_cartographer(
        question=question,
        graph=graph,
        schema_texts=schema_texts,
        schema_ids=schema_ids,
        embedder=embedder,
        faiss_index=faiss_index,
        doc_tokens=doc_tokens,
        df=df_stats
    )


In [11]:
class ErrorType:
    JOIN = "join_error"
    LOGIC = "logic_error"
    EXECUTION = "execution_error"
    SEMANTIC = "semantic_error"
    UNKNOWN = "unknown_error"


In [12]:
def classify_failure(verdict: Dict[str, Any]) -> str:
    reason = (verdict.get("reason") or "").lower()

    if "join" in reason or "table" in reason:
        return ErrorType.JOIN
    if "semantic" in reason or "metric" in reason:
        return ErrorType.SEMANTIC
    if "timeout" in reason or "execution" in reason:
        return ErrorType.EXECUTION
    if "sql" in reason:
        return ErrorType.LOGIC

    return ErrorType.UNKNOWN


In [13]:
def sql_fingerprint(sql: str) -> str:
    return hashlib.sha256(sql.strip().lower().encode()).hexdigest()


def has_converged(history: List[str], k: int = 2) -> bool:
    if len(history) < k:
        return False
    return history[-k:] == [history[-1]] * k


In [14]:
class RetryBudget:
    def __init__(self, max_steps: int):
        self.max_steps = max_steps
        self.used = 0

    def allow(self) -> bool:
        if self.used >= self.max_steps:
            return False
        self.used += 1
        return True


In [15]:
from llama_cpp import Llama
from pathlib import Path

MODELS_DIR = Path("../models")

llm_large = Llama(
    model_path=str(MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
    verbose=True
)

llm_medium = Llama(
    model_path=str(MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    n_ctx=2048,
    n_threads=8,
    n_batch=256,
    verbose=True
)

print("✅ Architect LLMs loaded")


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


✅ Architect LLMs loaded


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [16]:
from typing import Dict, Any, List
from pathlib import Path
import hashlib

def reflex_orchestrator(
    question: str,
    schema_text: str,
    db_path: Path,

    graph,
    schema_texts,
    schema_ids,
    embedder,
    faiss_index,
    doc_tokens,
    df_stats,

    llm_large,
    llm_medium,

    max_retries: int = 5,
    min_confidence: float = 0.6
) -> Dict[str, Any]:

    from agents.cartographer import graphrag_cartographer
    from agents.architect import architect_ensemble
    from agents.verifier import execution_verifier
    from agents.critic import semantic_verifier

    seen = set()
    retries = 0
    schema_context = None

    while retries < max_retries:
        retries += 1

        # 1️⃣ CARTOGRAPHER
        if schema_context is None:
            schema_context = graphrag_cartographer(
                question=question,
                graph=graph,
                schema_texts=schema_texts,
                schema_ids=schema_ids,
                embedder=embedder,
                faiss_index=faiss_index,
                doc_tokens=doc_tokens,
                df=df_stats
            )

        # 2️⃣ ARCHITECT
        candidates = architect_ensemble(
            question=question,
            schema=schema_text,
            llm_large=llm_large,
            llm_medium=llm_medium
        )

        if not candidates:
            return {"status": "failed", "reason": "No SQL generated"}

        for cand in candidates:
            sql = cand["sql"]
            intent = cand["intent"]

            fp = hashlib.sha256(sql.lower().encode()).hexdigest()
            if fp in seen:
                return {"status": "failed", "reason": "Convergence detected"}
            seen.add(fp)

            exec_v = execution_verifier(
                sql=sql,
                intent=intent,
                db_path=db_path
            )

            if not exec_v["allowed"]:
                schema_context = None
                continue

            sem_v = semantic_verifier(
                question=question,
                sql=sql,
                df=exec_v.get("result_df"),
                min_confidence=min_confidence
            )

            if sem_v["ok"]:
                return {
                    "status": "success",
                    "sql": sql,
                    "confidence": sem_v["confidence"],
                    "rows": exec_v.get("rows"),
                    "latency": exec_v.get("latency")
                }

        schema_context = None

    return {"status": "failed", "reason": "Retry budget exhausted"}



In [17]:
result = reflex_orchestrator(
    question="List student names and course titles",
    schema_text="""
tables:
- student(student_id, name)
- course(course_id, title)
- enrollment(student_id, course_id)
""",
    db_path=Path("../data/spider/database/academic/academic.sqlite"),

    graph=graph,
    schema_texts=schema_texts,
    schema_ids=schema_ids,
    embedder=embedder,
    faiss_index=faiss_index,
    doc_tokens=doc_tokens,
    df_stats=df_stats,

    llm_large=llm_large,
    llm_medium=llm_medium,

    max_retries=5
)

print(result)




Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'status': 'failed', 'reason': 'Convergence detected'}
